# Lección 9.8 — Regularización en redes: entrenar para el examen, no para memorizar las respuestas

> **Módulo 9 · Redes neuronales desde la intuición** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 4.4 (sobreajuste vs. subajuste), Lección 7.1 (regularización L1/L2), Lección 7.2 (early stopping), Lección 9.4 (bucle de entrenamiento), Lección 9.7 (inicialización)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Reconocer el **sobreajuste** de una red en sus curvas de entrenamiento y validación.
- Explicar con una analogía cómo funcionan **dropout**, **weight decay**, **batch normalization**, **data augmentation** y **early stopping**.
- Aplicar cada técnica en PyTorch y comparar su efecto.
- Usar correctamente `model.train()` y `model.eval()`, y explicar por qué importan.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Usaremos **MNIST** (dígitos escritos a mano de 28×28 píxeles). Para provocar sobreajuste a propósito, entrenaremos con **solo 1,000 imágenes** y validaremos con otras 5,000. La primera vez se descargan unos 10 MB.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets, transforms

torch.manual_seed(0)
rng = np.random.default_rng(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (8, 4.5)

mnist = datasets.MNIST("data", train=True, download=True)
orden = torch.randperm(60_000, generator=torch.Generator().manual_seed(0))
X = mnist.data.float() / 255            # píxeles entre 0 y 1
y = mnist.targets
X_tr, y_tr = X[orden[:1000]], y[orden[:1000]]            # 1,000 para entrenar (¡pocas!)
X_va, y_va = X[orden[1000:6000]], y[orden[1000:6000]]    # 5,000 para validar
MEDIA, DESV = 0.1307, 0.3081                             # estandarización habitual de MNIST (Lección 2.3)
print(f"✅ Todo listo · entrenamiento {tuple(X_tr.shape)} · validación {tuple(X_va.shape)} · dispositivo: {device}")

---
## 1. 📖 La intuición: el estudiante que memoriza

Dos estudiantes preparan un examen con **la misma guía de 100 preguntas resueltas**:
- **Memo** se aprende las 100 respuestas de memoria. En la guía saca 100. En el examen, con preguntas nuevas, se hunde.
- **Ana** entiende *por qué* cada respuesta es correcta. En la guía saca 92. En el examen, 90.

Una red neuronal con millones de perillas es un **Memo en potencia**: tiene capacidad de sobra para memorizar todos los ejemplos de entrenamiento, incluido su ruido (Lección 4.4). **Regularizar** es ponerle obstáculos a la memorización para obligarla a aprender lo general, como Ana.

Ya conoces dos formas de regularizar de módulos anteriores: castigar los pesos grandes (L2, Lección 7.1) y parar a tiempo (early stopping, Lección 7.2). Las redes las usan, y añaden trucos propios.

### ✋ Ejercicio 1 — ¿Quién memoriza?
Tres redes tienen estas exactitudes. ¿Cuál está sobreajustando, cuál subajustando y cuál está sana?

| Red | Entrenamiento | Validación |
|---|---|---|
| A | 100 % | 87 % |
| B | 71 % | 70 % |
| C | 94 % | 93 % |

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **A:** sobreajuste (brecha de 13 puntos: memorizó).
- **B:** subajuste (le va mal en las dos: el modelo se queda corto, Lección 4.4).
- **C:** sana (bien en las dos, brecha pequeña).

La regularización es la medicina para **A**. Darle regularización a **B** la empeoraría.
</details>

---
## 2. 📖 El paciente: una red grande con pocos datos

Nuestra red tiene **dos capas ocultas de 512 neuronas**: unos 670,000 pesos para aprender de 1,000 imágenes. Es una receta para memorizar. Primero definimos las herramientas: una función que construye la red (con las opciones de regularización que iremos activando) y el bucle de entrenamiento de la Lección 9.4.

In [ ]:
def crear_red(dropout=0.0, batchnorm=False):
    capas, entrada = [nn.Flatten()], 28 * 28
    for neuronas in (512, 512):
        capas.append(nn.Linear(entrada, neuronas))
        if batchnorm:
            capas.append(nn.BatchNorm1d(neuronas))     # sección 5
        capas.append(nn.ReLU())
        if dropout > 0:
            capas.append(nn.Dropout(dropout))          # sección 3
        entrada = neuronas
    capas.append(nn.Linear(entrada, 10))
    return nn.Sequential(*capas)

aumento = transforms.RandomAffine(degrees=10, translate=(0.1, 0.1))   # sección 6

def entrenar(red, weight_decay=0.0, aumentar=False, epocas=40):
    opt = torch.optim.AdamW(red.parameters(), lr=1e-3, weight_decay=weight_decay)   # sección 4
    historia = {"train": [], "val": [], "acc_train": [], "acc_val": []}
    for _ in range(epocas):
        red.train()                                     # modo entrenamiento (sección 7)
        orden = torch.randperm(len(X_tr))
        for i in range(0, len(X_tr), 64):
            idx = orden[i:i + 64]
            imagenes = X_tr[idx]
            if aumentar:                                # cada imagen se mueve/gira un poco al azar
                imagenes = torch.stack([aumento(img[None])[0] for img in imagenes])
            opt.zero_grad()
            perdida = nn.functional.cross_entropy(red((imagenes - MEDIA) / DESV), y_tr[idx])
            perdida.backward()
            opt.step()
        red.eval()                                      # modo evaluación para medir
        with torch.no_grad():
            for nombre, Xs, ys in [("train", X_tr, y_tr), ("val", X_va, y_va)]:
                salida = red((Xs - MEDIA) / DESV)
                historia[nombre].append(nn.functional.cross_entropy(salida, ys).item())
                historia["acc_" + nombre].append((salida.argmax(1) == ys).float().mean().item())
    return historia

def graficar(historias):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for nombre, h in historias.items():
        linea, = axes[0].plot(h["val"], label=f"{nombre} (val)")
        axes[0].plot(h["train"], "--", color=linea.get_color(), alpha=.6)
        axes[1].plot(h["acc_val"], label=nombre, color=linea.get_color())
    axes[0].set_title("Pérdida (continua = validación, punteada = entrenamiento)"); axes[0].set_ylim(0, 1)
    axes[1].set_title("Exactitud en validación"); axes[1].set_ylim(.8, 1)
    for ax in axes: ax.set_xlabel("época"); ax.legend(fontsize=8); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

def resumen(nombre, h):
    mejor = int(np.argmin(h["val"]))
    print(f"{nombre:<22} final → train {h['acc_train'][-1]:.1%} · val {h['acc_val'][-1]:.1%} · pérdida val {h['val'][-1]:.3f}"
          f"   | mejor pérdida val {h['val'][mejor]:.3f} en la época {mejor + 1}")

### 🔮 Ejercicio 2 — Las curvas del paciente
Entrenaremos la red **sin regularización** durante 40 épocas. Antes de ejecutar, dibuja en papel cómo esperas que se vean:
1. la pérdida de **entrenamiento**,
2. la pérdida de **validación** (pista: Lección 3.7).

✍️ *Mi dibujo / predicción:*

In [ ]:
torch.manual_seed(0)
red_base = crear_red()
h_base = entrenar(red_base)
resumen("sin regularización", h_base)
graficar({"sin regularización": h_base})

<details><summary><b>👀 Ver respuesta</b></summary>

- La pérdida de **entrenamiento** cae casi a **0** y la exactitud de entrenamiento llega al **100 %**: memorizó las 1,000 imágenes.
- La pérdida de **validación** baja solo al principio: alcanza su mínimo hacia la **época 7** (≈0.42) y después **sube** (≈0.54 en la época 40), aunque la exactitud de validación se queda estancada (≈90 %). La red se vuelve cada vez más **confiada** en sus errores: por eso la pérdida empeora aunque la exactitud apenas se mueva.

Esa forma de "U" (o de "palo de hockey") en la validación es la **firma del sobreajuste**. Ahora probaremos las medicinas una por una.
</details>

---
## 3. 📖 Dropout: entrenar con un equipo incompleto

Imagina un equipo de fútbol donde, en cada entrenamiento, el entrenador **manda a la banca a la mitad de los jugadores al azar**. Nadie puede depender de que "el crack" haga todo: cada jugador aprende a ser útil por su cuenta y a cooperar con cualquier combinación de compañeros. El día del partido juegan **todos**.

**Dropout** hace eso con las neuronas: en cada paso de entrenamiento, **apaga al azar** una fracción `p` de ellas (pone su salida en 0). Así la red no puede construir "conspiraciones" frágiles entre neuronas que solo sirven para memorizar.

Veámoslo **a mano con numpy** en una capa de 10 neuronas:

In [ ]:
activaciones = np.round(rng.uniform(0.5, 2.0, 10), 2)
p = 0.5                                                   # ✏️ fracción que se apaga

mascara = rng.random(10) > p                              # True = la neurona juega, False = a la banca
con_dropout = activaciones * mascara / (1 - p)            # se reescala para mantener el promedio

print("activaciones  :", activaciones)
print("máscara       :", mascara.astype(int))
print("con dropout   :", np.round(con_dropout, 2))
print(f"\nsuma original: {activaciones.sum():.2f} · suma con dropout: {con_dropout.sum():.2f}")

### ✋ Ejercicio 3 — ¿Por qué dividir entre (1 − p)?
Fíjate en que las neuronas que sobreviven se **multiplican por 2** (porque 1 / (1 − 0.5) = 2). ¿Para qué? Pista: piensa en qué pasa el "día del partido", cuando juegan todas.

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Para que el **volumen total** de la señal sea el mismo, en promedio, durante el entrenamiento (con la mitad apagada) y durante la predicción (con todas encendidas). Si no se reescalara, la siguiente capa se acostumbraría a recibir la mitad de señal y el día del partido recibiría el doble. Las sumas impresas arriba no coinciden exactamente porque la máscara es aleatoria, pero **en promedio** sí.

En PyTorch, `nn.Dropout(p)` hace todo esto solo, y **solo en modo entrenamiento** (sección 7).
</details>

### 🔮 Ejercicio 4 — Dropout contra el sobreajuste
Activamos `dropout=0.5` después de cada capa oculta. Predice: ¿la pérdida de **entrenamiento** bajará más rápido o más lento que antes? ¿Y qué pasará con la curva de validación?

✍️ *Mi predicción:*

In [ ]:
DROPOUT = 0.5      # ✏️ prueba 0.2 y 0.8

torch.manual_seed(0)
h_dropout = entrenar(crear_red(dropout=DROPOUT))
resumen("sin regularización", h_base)
resumen(f"dropout {DROPOUT}", h_dropout)
graficar({"sin regularización": h_base, f"dropout {DROPOUT}": h_dropout})

<details><summary><b>👀 Ver respuesta</b></summary>

- La pérdida de **entrenamiento** baja **más lento** (la red entrena "con medio equipo") y la exactitud de entrenamiento se queda un pelo por debajo del 100 % (≈99.7 %).
- El mínimo de la pérdida de validación es **más bajo** que sin regularizar (≈0.37 contra ≈0.42), pero la curva sigue teniendo forma de U y la exactitud final de validación queda prácticamente igual (≈90 %).

Con solo 1,000 imágenes y una red tan grande, el dropout **ayuda poco por sí solo**: la red sigue teniendo capacidad de sobra para memorizar. ✏️ Con 0.8 el entrenamiento se vuelve mucho más lento; con 0.2 el efecto casi no se nota. No hay medicina milagrosa: hay que medir.
</details>

---
## 4. 📖 Weight decay: impuesto a los pesos grandes

Es la regularización **L2** de la Lección 7.1 aplicada a la red: en cada paso, cada peso se **encoge un poquito** hacia cero, como un impuesto que se cobra en proporción a lo grande que es. Solo los pesos que de verdad ayudan a bajar la pérdida "ganan" lo suficiente para mantenerse grandes. Una red con pesos pequeños produce funciones más suaves, que memorizan menos.

> 📝 **¿Adam o AdamW?** Con Adam, el parámetro `weight_decay` se mezcla con las escalas adaptativas del optimizador y pierde fuerza. **AdamW** (la "W" es de *weight decay* desacoplado) aplica el impuesto por separado y funciona como se espera. Por eso nuestro bucle usa `AdamW`: con `weight_decay=0` es exactamente Adam.

### 🔮 Ejercicio 5 — ¿Cuánto impuesto?
Probaremos `weight_decay` de **0.1** y de **1.0**. ¿Qué crees que pasará con el tamaño total de los pesos? ¿Y con la curva de validación?

✍️ *Mi predicción:*

In [ ]:
def tamano_pesos(red):
    return sum(p.norm() ** 2 for p in red.parameters()).sqrt().item()

resultados_wd = {}
for wd in [0.1, 1.0]:                     # ✏️ prueba también 0.01 o 10
    torch.manual_seed(0)
    red = crear_red()
    resultados_wd[f"weight decay {wd}"] = entrenar(red, weight_decay=wd)
    print(f"weight_decay={wd:<4} → tamaño total de los pesos al final: {tamano_pesos(red):.1f}")

print(f"sin weight decay  → tamaño total de los pesos al final: {tamano_pesos(red_base):.1f}\n")

resumen("sin regularización", h_base)
for nombre, h in resultados_wd.items():
    resumen(nombre, h)
graficar({"sin regularización": h_base, **resultados_wd})

<details><summary><b>👀 Ver respuesta</b></summary>

- El tamaño de los pesos **baja** cuanto más alto es el impuesto (compara las tres líneas impresas: sin weight decay es el más grande y con 1.0 el más pequeño).
- Con **0.1** el efecto es casi imperceptible (pérdida de validación final ≈0.51 contra ≈0.54).
- Con **1.0** la pérdida de validación **deja de subir**: su mínimo llega casi al final (época ≈37) y termina en ≈0.38. La "U" desaparece porque la red ya no puede volverse exageradamente confiada.
- Pero la **exactitud** de validación casi no cambia (≈90.5 %). El weight decay controla la *confianza* de la red, pero no le da información nueva: con 1,000 imágenes, el techo sigue ahí.
</details>

---
## 5. 📖 Batch normalization: un técnico de sonido en cada capa

En la Lección 9.7 vimos que el "volumen" de las activaciones puede desbocarse de capa en capa. **Batch normalization** (BatchNorm) pone un **técnico de sonido** después de cada capa: mira el lote actual de ejemplos y **re-normaliza** cada neurona para que tenga media 0 y desviación 1 (lo mismo que el escalado de la Lección 2.3, pero dentro de la red). Luego aplica una escala y un desplazamiento **aprendidos**, por si a la red le conviene otro volumen.

Su efecto principal es que el entrenamiento sea **más estable y más rápido** (permite tasas de aprendizaje más altas). Como cada lote es un poco distinto, añade algo de "ruido" que, de rebote, regulariza un poco.

A mano con numpy, para un lote de 4 ejemplos y 3 neuronas:

In [ ]:
lote = np.array([[ 10.,  0.2, -300.],
                 [ 12.,  0.1, -100.],
                 [  8.,  0.4, -200.],
                 [ 14.,  0.3, -400.]])

normalizado = (lote - lote.mean(axis=0)) / (lote.std(axis=0) + 1e-5)
print("Antes   → media por neurona:", lote.mean(axis=0).round(2), " desviación:", lote.std(axis=0).round(2))
print("Después → media por neurona:", normalizado.mean(axis=0).round(2), " desviación:", normalizado.std(axis=0).round(2))

### 🔮 Ejercicio 6 — ¿Regulariza tanto como dropout?
Añadimos `BatchNorm1d` después de cada capa lineal (sin dropout). Predice: comparada con la red sin regularización, ¿la pérdida de entrenamiento bajará más rápido o más lento? ¿Desaparecerá el sobreajuste?

✍️ *Mi predicción:*

In [ ]:
torch.manual_seed(0)
h_bn = entrenar(crear_red(batchnorm=True))
resumen("sin regularización", h_base)
resumen("batchnorm", h_bn)
graficar({"sin regularización": h_base, "batchnorm": h_bn})

<details><summary><b>👀 Ver respuesta</b></summary>

- La pérdida de entrenamiento baja **más rápido**: el mínimo de validación llega en la **época 4** y es el más bajo hasta ahora (≈0.33).
- Pero el entrenamiento sigue llegando al **100 %** y la exactitud de validación queda en ≈90.5 %, casi igual. **No cura el sobreajuste**: BatchNorm es, sobre todo, un **acelerador y estabilizador**. Su gran utilidad aparece en redes **profundas** (Lección 9.7 y Módulo 10), donde sin ella costaría mucho entrenar.
</details>

---
## 6. 📖 Data augmentation: más datos gratis

La mejor cura contra el sobreajuste es **tener más datos**. Si no los tienes, puedes **fabricarlos**: un 7 un poco girado o desplazado **sigue siendo un 7**. *Data augmentation* (aumento de datos) crea, en cada época, versiones ligeramente distintas de cada imagen. La red nunca ve dos veces exactamente la misma foto, así que memorizar se vuelve mucho más difícil.

Nuestro aumento gira hasta ±10° y desplaza hasta un 10 % (unos 3 píxeles). Mira 6 versiones aumentadas de la misma imagen:

In [ ]:
torch.manual_seed(1)
fig, axes = plt.subplots(1, 7, figsize=(12, 2))
axes[0].imshow(X_tr[0], cmap="gray"); axes[0].set_title("original")
for ax in axes[1:]:
    ax.imshow(aumento(X_tr[0][None])[0], cmap="gray"); ax.set_title("aumentada")
for ax in axes: ax.axis("off")
plt.show()

### ✋ Ejercicio 7 — Aumentos con sentido
Para cada problema, ¿el aumento propuesto es buena idea? ¿Por qué?
1. Dígitos escritos a mano → **voltear horizontalmente** (efecto espejo).
2. Fotos de perros y gatos → voltear horizontalmente.
3. Radiografías de tórax → girar 180°.
4. Fotos de productos → cambiar un poco el brillo.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. ❌ Un 2 o un 5 en espejo **ya no es el mismo dígito** (y algunos parecerían otro). El aumento debe **conservar la etiqueta**.
2. ✅ Un gato visto en espejo sigue siendo un gato.
3. ❌ Las radiografías siempre se toman en la misma orientación; una al revés no aparecerá nunca en la vida real, y el corazón cambiaría de lado.
4. ✅ Simula fotos tomadas con distinta luz.

Regla: **aumenta solo con transformaciones que podrían ocurrir en los datos reales y que no cambian la respuesta.** En el Módulo 10 (visión) verás muchos más aumentos.
</details>

### 🔮 Ejercicio 8 — El efecto del aumento
Entrenamos con aumento de datos (esta celda tarda algo más, porque transforma cada imagen en cada época). ¿Llegará la exactitud de **entrenamiento** al 100 % como antes?

✍️ *Mi predicción:*

In [ ]:
torch.manual_seed(0)
h_aug = entrenar(crear_red(), aumentar=True)
resumen("sin regularización", h_base)
resumen("aumento de datos", h_aug)
graficar({"sin regularización": h_base, "aumento de datos": h_aug})

<details><summary><b>👀 Ver respuesta</b></summary>

**No.** La exactitud de entrenamiento se queda en ≈98.7 %: la red ya no puede memorizar porque nunca ve la misma imagen dos veces. Y la validación da el **mayor salto** de toda la lección: ≈94.4 % (contra ≈90.3 %), con la pérdida de validación bajando casi hasta el final (≈0.21) en vez de formar una U.

Esto es típico: cuando hay pocos datos, **fabricar variaciones realistas** suele ser la medicina más potente, porque es la única que le da a la red **información nueva** (que un 7 girado sigue siendo un 7).
</details>

---
## 7. 📖 `model.train()` y `model.eval()`: el entrenamiento y el partido

Dropout y BatchNorm se comportan **distinto** al entrenar y al predecir:

| Capa | En `train()` | En `eval()` |
|---|---|---|
| `Dropout` | Apaga neuronas al azar | No apaga nada (juegan todos) |
| `BatchNorm` | Normaliza con la media/desviación **del lote actual** y actualiza un promedio acumulado | Usa el promedio **acumulado** durante el entrenamiento |

Por eso nuestro bucle llama a `red.train()` antes de entrenar y a `red.eval()` antes de medir. Olvidarlo es uno de los errores más comunes (y silenciosos) de PyTorch.

### 🔮 Ejercicio 9 — La misma imagen, cinco veces
Tomamos la red con dropout ya entrenada y le pasamos **la misma imagen** 5 veces en cada modo. Predice: ¿saldrán las mismas probabilidades las 5 veces?

✍️ *Mi predicción:* en `train()` → ___ · en `eval()` → ___

In [ ]:
torch.manual_seed(0)
red_dropout = crear_red(dropout=0.5)
entrenar(red_dropout, epocas=10)
imagen = (X_va[:1] - MEDIA) / DESV

for modo in ["train", "eval"]:
    red_dropout.train() if modo == "train" else red_dropout.eval()
    with torch.no_grad():
        probs = [torch.softmax(red_dropout(imagen), 1)[0, y_va[0]].item() for _ in range(5)]
    print(f"modo {modo:<5} → probabilidad de la clase correcta ({y_va[0].item()}): {np.round(probs, 3)}")

red_dropout.train()
with torch.no_grad():
    acc_train_modo = (red_dropout((X_va - MEDIA) / DESV).argmax(1) == y_va).float().mean().item()
red_dropout.eval()
with torch.no_grad():
    acc_eval_modo = (red_dropout((X_va - MEDIA) / DESV).argmax(1) == y_va).float().mean().item()
print(f"\nExactitud en validación medida en modo train: {acc_train_modo:.1%} · en modo eval: {acc_eval_modo:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- En **`train()`** las 5 probabilidades son **distintas** (entre ≈0.97 y ≈0.999): en cada pasada el dropout apaga neuronas diferentes.
- En **`eval()`** las 5 son **idénticas** (≈0.986): no se apaga nada.
- Medida en modo `train`, la exactitud de validación sale ≈87 %; en modo `eval`, ≈90 %. Si olvidas `red.eval()`, tus métricas serán **más bajas y más ruidosas** de lo que la red realmente es. Y con BatchNorm puede ser peor: en modo `train` las predicciones de un ejemplo dependen de los **demás** ejemplos del lote.
</details>

---
## 8. 📖 Early stopping y comparación final

La medicina más simple: **guardar la red en la época en que la validación fue mejor** y quedarse con esa (Lección 7.2). En las gráficas anteriores, el mínimo de la pérdida de validación de la red sin regularizar ocurrió muy pronto: si hubiéramos parado ahí, nos habríamos ahorrado el empeoramiento posterior.

```python
# Patrón típico de early stopping con paciencia
mejor, sin_mejorar, PACIENCIA = float("inf"), 0, 5
for epoca in range(200):
    entrenar_una_epoca()
    perdida_val = evaluar()
    if perdida_val < mejor:
        mejor, sin_mejorar = perdida_val, 0
        torch.save(red.state_dict(), "mejor_red.pt")   # guardar la mejor versión
    else:
        sin_mejorar += 1
        if sin_mejorar >= PACIENCIA:
            break                                       # llevamos 5 épocas sin mejorar: parar
red.load_state_dict(torch.load("mejor_red.pt"))
```

### 🔮 Ejercicio 10 — El combo
Combinemos **dropout 0.5 + weight decay 0.1 + aumento de datos**. ¿Será mejor que cada técnica por separado? ¿Cuál es la mejor exactitud de validación que crees que alcanzará?

✍️ *Mi predicción:*

In [ ]:
torch.manual_seed(0)
h_combo = entrenar(crear_red(dropout=0.5), weight_decay=0.1, aumentar=True)

todas = {"sin regularización": h_base, f"dropout {DROPOUT}": h_dropout, **resultados_wd,
         "batchnorm": h_bn, "aumento de datos": h_aug, "combo": h_combo}
for nombre, h in todas.items():
    resumen(nombre, h)
graficar({"sin regularización": h_base, "combo": h_combo})

<details><summary><b>👀 Ver respuesta</b></summary>

El combo llega a ≈93.2 % de validación: mucho mejor que sin regularizar (≈90.3 %), pero **peor que el aumento de datos solo** (≈94.4 %). Fíjate en su exactitud de entrenamiento: ≈96.7 %. Con tres frenos a la vez, la red empieza a quedarse **corta** (un toque de subajuste).

Lección importante: **las regularizaciones no se suman sin más**. Cada técnica quita algo de capacidad; si ya quitaste suficiente, añadir más empeora. Por eso se ajustan **una a una**, midiendo con validación (Lección 7.4).
</details>

---
## 9. 📖 Las perillas de la regularización

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `p` de `nn.Dropout` | Fracción de neuronas apagadas en cada paso | Más regularización; entrena más lento; si es muy alto, subajuste | Menos regularización (0 = sin dropout) | 0.1–0.3 en redes grandes modernas; 0.5 en capas densas de redes con pocos datos |
| `weight_decay` (AdamW) | Impuesto a los pesos grandes | Pesos más pequeños, más simple; muy alto → subajuste | Más libertad para memorizar | 0.01 (rango 1e-4 a 0.1) |
| `BatchNorm1d/2d` | Re-normalizar activaciones por lote | — (se pone o no se pone) | — | Úsalo en redes profundas; ojo con lotes muy pequeños (< 16) |
| `momentum` de BatchNorm | Velocidad con que se actualiza el promedio acumulado | Promedio más "nervioso" | Promedio más lento | 0.1 (valor por defecto de PyTorch) |
| Intensidad del aumento (grados, desplazamiento) | Cuánto se deforman los ejemplos | Más variedad; demasiado → imágenes irreconocibles, subajuste | Menos efecto | Pequeño y realista (±10°, 10 %) |
| Paciencia del early stopping | Épocas sin mejorar antes de parar | Más tolerante al ruido de la curva | Para antes (riesgo de parar en un bache) | 5–10 épocas |

### 🧠 Regla práctica
> **Primero consigue sobreajustar** (una red capaz de memorizar), **después regulariza**. Si tu red ni siquiera aprende bien el entrenamiento, el problema es otro (Lección 9.9), y regularizar lo empeorará.

---
## 10. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Entrenamiento ~100 %, validación muy por debajo, pérdida de validación sube | Sobreajuste | Más datos o aumento, dropout, weight decay, early stopping, red más pequeña |
| Entrenamiento y validación malos, y empeoran al regularizar | Subajuste: sobra regularización | Bajar `p` / `weight_decay`, red más grande |
| La validación cambia mucho entre dos evaluaciones del mismo modelo | Olvidaste `model.eval()` (dropout activo al medir) | `red.eval()` antes de validar, `red.train()` antes de entrenar |
| Error o resultados raros al predecir **un solo ejemplo** con BatchNorm | BatchNorm en modo `train` necesita un lote para calcular la media | `red.eval()` antes de predecir |
| La validación es **mejor** que el entrenamiento | Normal con dropout/aumento si mides el entrenamiento en modo train | Mide las dos en modo `eval()` para compararlas de forma justa |
| `weight_decay` en Adam no parece hacer nada | Adam mezcla el decay con su escala adaptativa | Usar `AdamW` |

### 🔮 Ejercicio 11 — Diagnostica
Un colega entrena una red con dropout 0.8, weight decay 0.5 y aumento de datos fuerte (giros de ±60°). Obtiene 62 % en entrenamiento y 61 % en validación. Te pide **más** regularización "porque la red no generaliza". ¿Qué le dices?

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver respuesta</b></summary>

Que su red **generaliza perfectamente** (62 % vs. 61 %: no hay brecha): lo que tiene es **subajuste**. Le sobra regularización. Debería **quitar** freno: dropout 0.2–0.3, weight decay ~0.01, aumentos más suaves (con ±60° muchos dígitos se vuelven otros). Recuerda la tabla del Ejercicio 1: regularizar a la red B la empeora.
</details>

---
## 11. 📋 Ficha: regularización en redes

| Técnica | Analogía | Qué hace | En PyTorch |
|---|---|---|---|
| **Dropout** | Entrenar con medio equipo en la banca | Apaga neuronas al azar al entrenar | `nn.Dropout(p)` |
| **Weight decay** | Impuesto a los pesos grandes | Encoge los pesos en cada paso (L2) | `torch.optim.AdamW(..., weight_decay=0.01)` |
| **Batch normalization** | Técnico de sonido en cada capa | Re-normaliza activaciones por lote; estabiliza | `nn.BatchNorm1d(n)` / `nn.BatchNorm2d(n)` |
| **Data augmentation** | Estudiar con preguntas parecidas pero nuevas | Crea variaciones realistas de los ejemplos | `torchvision.transforms` |
| **Early stopping** | Entregar el examen cuando mejor ibas | Se queda con la época de mejor validación | Bucle con paciencia + `state_dict` |

## 12. 🧠 Resumen en 5 puntos
1. Una red grande con pocos datos **memoriza**: la pérdida de validación forma una "U" mientras la de entrenamiento llega a 0.
2. **Dropout** apaga neuronas al azar al entrenar; **weight decay** (mejor con AdamW) encoge los pesos.
3. **BatchNorm** re-normaliza cada capa: sobre todo estabiliza y acelera; regulariza solo un poco.
4. **Aumentar los datos** con transformaciones realistas suele ser la medicina más potente, y **early stopping** es la más barata.
5. Dropout y BatchNorm cambian entre `train()` y `eval()`: mide siempre en modo `eval()`.

---
## 13. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 13.1 Repaso rápido (opción múltiple)

**1.** ¿Qué hace `nn.Dropout(0.3)` cuando la red está en modo `eval()`?
- a) Apaga el 30 % de las neuronas · b) Apaga el 70 % · c) Nada · d) Multiplica todo por 0.3

**2.** Tu red tiene 99 % en entrenamiento y 80 % en validación. ¿Qué cambio tiene **menos** sentido?
- a) Añadir dropout · b) Subir `weight_decay` · c) Duplicar el número de neuronas · d) Aumentar los datos

**3.** ¿Por qué se prefiere `AdamW` a `Adam` cuando se usa weight decay?
- a) Es más rápido · b) Aplica el weight decay por separado de la escala adaptativa · c) No necesita tasa de aprendizaje · d) Tiene dropout incorporado

**4.** Para clasificar **letras** escritas a mano, ¿cuál es un mal aumento de datos?
- a) Girar ±5° · b) Desplazar 2 píxeles · c) Voltear horizontalmente · d) Cambiar un poco el grosor del trazo

**5.** ¿Cuál es el efecto **principal** de BatchNorm?
- a) Reducir el tamaño de la red · b) Estabilizar y acelerar el entrenamiento · c) Aumentar los datos · d) Sustituir a la función de pérdida

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** En `eval()` el dropout no apaga nada (y no hace falta reescalar: ya se reescaló durante el entrenamiento).
2. **c)** Más neuronas = más capacidad de memorizar; las demás son medicinas contra el sobreajuste.
3. **b)** El "desacoplamiento" es la clave de AdamW.
4. **c)** Una "b" en espejo es una "d": el aumento cambiaría la etiqueta.
5. **b)** Su regularización es un efecto secundario.
</details>

### 13.2 Preguntas para pensar (sin ejecutar)

**A.** El dropout se parece a entrenar **muchas redes distintas** (una por cada máscara aleatoria) que comparten pesos. ¿Con qué técnica del Módulo 6 lo relacionarías? ¿Por qué eso ayudaría a generalizar?

**B.** Un equipo tiene 5 millones de fotos de productos y una red mediana. Su validación es igual a su entrenamiento (89 % vs. 89 %). Alguien propone dropout 0.5. ¿Buena idea?

**C. Caso 💼.** Una startup médica tiene solo 800 radiografías etiquetadas y una red que sobreajusta. Tiene presupuesto para **una** de estas opciones: (1) pagar a radiólogos para etiquetar 4,000 imágenes más, o (2) contratar a un ingeniero que pruebe dropout, weight decay y aumento de datos durante un mes. ¿Qué recomendarías y por qué?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Con los **ensambles** (bagging, Lección 6.2): promediar muchos modelos distintos reduce la varianza. En `eval()` la red completa se comporta, aproximadamente, como el promedio de todas esas sub-redes. Cada sub-red memoriza cosas distintas, y al promediarlas el ruido se cancela.

**B.** Probablemente no: no hay brecha, así que no hay sobreajuste que curar. Con tantos datos, si quieren mejorar deberían **aumentar la capacidad** (red más grande, más épocas) y regularizar solo si aparece una brecha.

**C.** No hay una única respuesta; importa el razonamiento. Un buen argumento: **más datos reales** suelen ser la mejor medicina y su efecto es duradero; 800 → 4,800 imágenes multiplica por 6 la información. Las técnicas de regularización son baratas y deberían usarse **de todas formas** (son unas pocas líneas de código; no requieren un mes). Lo ideal: etiquetar más **y** aplicar regularización básica desde el día uno. Además, con datos médicos, la calidad de las etiquetas y la representatividad (hospitales, máquinas) importan tanto como la cantidad.
</details>

### 13.3 🧩 Práctica integradora: "la receta mínima"

Tu jefe quiere la **receta más simple** que dé un buen resultado con estas 1,000 imágenes. Lo harás en 4 pasos. **No avances sin revisar cada respuesta.**

**Paso 1 ✋ — Lee la tabla.** Usa la tabla de resultados del Ejercicio 10 (vuelve a ejecutar esa celda si hace falta). Ordena las técnicas por la **exactitud final de validación**. ¿Cuál fue la mejor por sí sola?

✍️ *Mi orden:*

<details><summary><b>👀 Ver respuesta</b></summary>

Por exactitud final de validación: **aumento de datos** (≈94.4 %) > combo (≈93.2 %) > batchnorm y weight decay 1.0 (≈90.5 %) > sin regularizar y weight decay 0.1 (≈90.3 %) > dropout 0.5 (≈89.8 %). La mejor por sí sola, con diferencia: **el aumento de datos**.
</details>

**Paso 2 ✏️ — Early stopping a mano.** La red **sin regularizar** alcanzó su mínimo de pérdida de validación muy pronto. Reentrénala solo hasta esa época y compara su exactitud de validación con la que tenía a las 40 épocas.

In [ ]:
EPOCA_DE_PARADA = 7     # ✏️ la época del mínimo que imprimió resumen("sin regularización", ...)

torch.manual_seed(0)
h_parada = entrenar(crear_red(), epocas=EPOCA_DE_PARADA)
resumen(f"parada en época {EPOCA_DE_PARADA}", h_parada)
resumen("sin regularización (40)", h_base)

<details><summary><b>👀 Ver respuesta</b></summary>

Parando en la época 7, la pérdida de validación es ≈0.42 (contra ≈0.54 a las 40 épocas): la red está mucho menos "sobreconfiada". Pero la **exactitud** de validación es ≈89.7 %, prácticamente igual que a las 40 épocas (≈90.3 %). Con este paciente, early stopping **ahorra tiempo** (7 épocas en vez de 40) y da probabilidades más sensatas, pero no sube la exactitud. Ojo: el criterio importa. Si paras según la pérdida, eliges la red mejor calibrada; si paras según la exactitud, la que más acierta. Decide antes cuál te importa.
</details>

**Paso 3 ✏️ — Tu receta.** Elige **una o dos** técnicas (las que te parezcan mejor relación beneficio/complejidad) y entrena tu receta. Intenta superar el 91 % de validación.

In [ ]:
MI_DROPOUT = 0.3        # ✏️
MI_WEIGHT_DECAY = 0.0   # ✏️
MI_AUMENTO = True       # ✏️

torch.manual_seed(0)
h_mia = entrenar(crear_red(dropout=MI_DROPOUT), weight_decay=MI_WEIGHT_DECAY, aumentar=MI_AUMENTO)
resumen("mi receta", h_mia)

<details><summary><b>👀 Ver respuesta</b></summary>

Con los valores propuestos (dropout 0.3 + aumento) se obtiene ≈93.9 %, muy parecido al aumento solo (≈94.4 %). Ambos superan con holgura el 91 %. La **receta mínima** es entonces **solo aumento de datos**: una línea de código y el mayor beneficio. Si probaste otras combinaciones, lo importante es que compares contra esa referencia y elijas la más simple entre las que empatan.
</details>

**Paso 4 ✋ — El informe.** Escribe tres frases para tu jefe: qué problema tenía la red, qué receta propones y qué ganarías si consiguieras más imágenes etiquetadas.

✍️ *Mi informe:*

<details><summary><b>👀 Ver un ejemplo</b></summary>

*"Con solo 1,000 imágenes, la red memorizaba los ejemplos de entrenamiento (100 % de acierto) pero solo acertaba el 90 % de las imágenes nuevas. La receta que propongo es generar, en cada vuelta del entrenamiento, versiones ligeramente giradas y desplazadas de cada imagen: con ese cambio de una línea pasamos a ≈94 %. Probamos otras técnicas más complejas y ninguna superó a esta. Si consiguiéramos más imágenes etiquetadas reales, esperaríamos mejorar todavía más, porque el problema de fondo es la falta de ejemplos."*
</details>

---
## 📚 Para profundizar
- 📄 **Artículo:** Srivastava et al. (2014), *Dropout: A Simple Way to Prevent Neural Networks from Overfitting* (muy legible).
- 📄 **Artículo:** Loshchilov y Hutter (2019), *Decoupled Weight Decay Regularization* (el origen de AdamW).
- 📺 **Video:** Andrej Karpathy, *A Recipe for Training Neural Networks* (blog, 2019): "primero sobreajusta, luego regulariza".
- 📘 **Documentación:** PyTorch, `nn.Dropout`, `nn.BatchNorm1d`, `torch.optim.AdamW`; torchvision, *Transforming and augmenting images*.

---
⬅️ **Anterior:** [9.7 Inicialización de pesos y gradientes que desaparecen o explotan: el teléfono descompuesto](9.7-inicializacion-y-gradientes.ipynb) · ➡️ **Siguiente:** [9.9 Recetario de diagnóstico: el consultorio de redes neuronales](9.9-recetario-de-diagnostico.ipynb)